In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/var/folders/lw/14dpn8b91z99svtdtfqnx09800tlp4/T/ipykernel_35067/3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [3]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [4]:
from sklearn.model_selection import train_test_split
X = data['text']
y = data['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)

## Data Preprocessing

In [5]:
# !pip install nltk

In [6]:
import string
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gnancy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [7]:
import re
def remove_html(html_string):
    return re.sub(r'<!--.*?-->', '', html_string, flags=re.DOTALL)

def remove_script(html_string):
    return re.sub(r'<(script|style).*?>.*?</\1>', '', html_string, flags=re.DOTALL | re.IGNORECASE)

def remove_tags(html_string):
    html_string = re.sub(r'<.*?>', '', html_string, flags=re.DOTALL )
    html_string = re.sub(r'\s+', ' ', html_string).strip()
    return html_string

data['text'] = data['text'].apply(remove_html).apply(remove_script).apply(remove_tags)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [8]:
def clean_text(text_string):
    text_string = text_string.lower()
    text_string = re.sub(r'[^a-z0-9\s]','',text_string)
    text_string = re.sub(r'\d+','',text_string)
    text_string = re.sub(r'\b[a-z]\b','',text_string)
    text_string = re.sub(r'\s+',' ',text_string).strip()
    if text_string.startswith('b'):
        text_string = text_string[2:]
    return text_string

data['text'] = data['text'].apply(clean_text)
data['text']

0      dear sir strictly private business proposal am...
1                                                will do
2      noracheryl has emailed dozens of memos about h...
3      dear sirfmadamc know that this proposal might ...
4                                                    fyi
                             ...                        
995    so whats the latest it sounds contradictory an...
996    transfer of million pounds to youraccountmy na...
997    rb will call to explain are you back in the co...
998       yang on travelnot free tonitemay work tomorrow
999    sbwhoeopsunday february pmhshaunh just talked ...
Name: text, Length: 1000, dtype: object

## Now let's work on removing stopwords
Remove the stopwords.

In [9]:
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

def text_remove_stop_words(test_str):
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(test_str)
    filtered_text = ' '.join([word for word in words if word.lower() not in stop_words])
    return filtered_text

data['text'] = data['text'].apply(text_remove_stop_words)
data['text']


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gnancy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/gnancy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


0      dear sir strictly private business proposal mi...
1                                                       
2      noracheryl emailed dozens memos haiti weekend ...
3      dear sirfmadamc know proposal might surprise e...
4                                                    fyi
                             ...                        
995    whats latest sounds contradictory af decide sh...
996    transfer million pounds youraccountmy name mre...
997                         rb call explain back country
998          yang travelnot free tonitemay work tomorrow
999    sbwhoeopsunday february pmhshaunh talked shaun...
Name: text, Length: 1000, dtype: object

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [10]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

lemmizer = WordNetLemmatizer()

def word_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J":wordnet.ADJ,
                "N":wordnet.NOUN,
                "V":wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def word_lemmatizer(text_str):
    words = word_tokenize(text_str)
    lemma_Words = [lemmizer.lemmatize(word, word_pos(word)) for word in words ]
    lemma_str = ' '.join(lemma_Words).strip()
    return lemma_str

data['text'] = data['text'].apply(word_lemmatizer)
data['text']

[nltk_data] Downloading package wordnet to /Users/gnancy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/gnancy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/gnancy/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


0      dear sir strictly private business proposal mi...
1                                                       
2      noracheryl email dozen memo haiti weekend plea...
3      dear sirfmadamc know proposal might surprise e...
4                                                    fyi
                             ...                        
995    whats late sound contradictory af decide shall...
996    transfer million pound youraccountmy name mrej...
997                         rb call explain back country
998          yang travelnot free tonitemay work tomorrow
999    sbwhoeopsunday february pmhshaunh talk shaun w...
Name: text, Length: 1000, dtype: object

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [11]:
from sklearn.model_selection import train_test_split
X = data['text']
y = data['label']
data_train, data_val, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)

In [12]:
from collections import Counter

txt = ' '.join(X)
words = word_tokenize(txt)
word_c = Counter(words).most_common(10)
word_c

[('money', 924),
 ('account', 801),
 ('bank', 754),
 ('fund', 710),
 ('u', 665),
 ('transfer', 529),
 ('business', 495),
 ('country', 457),
 ('contact', 447),
 ('transaction', 416)]

## Extra features

In [13]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€","$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train.str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train.str.contains(suspicious_words)*1
data_train['text_len'] = data_train.apply(lambda x: len(x)) 

data_val['money_mark'] = data_val.str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val.str.contains(suspicious_words)*1
data_val['text_len'] = data_val.apply(lambda x: len(x)) 

data_train.head()

970    mrsemaureen wusuecotonoucrepublic benindear fr...
137    dear friendthis letter must come big surprise ...
82                                release hold need call
188                                                     
358    fromamohammed abacha lagosnigeria dear sirfmad...
Name: text, dtype: object

## How would you create a Bag of Words with the CountVectorizer method?

In [14]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

vect = CountVectorizer()
X_train_cnt = vect.fit_transform(X_train).toarray()
X_test_cnt = vect.transform(X_test).toarray()
print(np.argmax(vect.transform(['money']).toarray()))

print(vect.get_feature_names_out()[:-10])

8770
['aac' 'aaclocated' 'aae' ... 'zimbabwethis' 'zimbabwewhile' 'zip']


## TD-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tdidf = TfidfVectorizer(
    max_features=2000,
    min_df=4,
    max_df=900,
    ngram_range=(1,2)
)

X_train_cnt_td = tdidf.fit_transform(X_train)
X_test_cnt_td = tdidf.transform(X_test)
print(X_train_cnt_td.get_shape())
X_train_cnt_td.toarray()[:5]

print(tdidf.get_feature_names_out()[:20])

(800, 2000)
['abacha' 'abandon' 'abandon sum' 'abidjan' 'abidjan cote' 'able' 'abroad'
 'abroad dont' 'absolute' 'ac' 'accept' 'acceptable' 'acceptance' 'access'
 'accident' 'accord' 'accord percentage' 'accordingly' 'account'
 'account abroad']


### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to find the **best feature representation**.

You can work with teams of two persons (recommended).

In [17]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report

model = MultinomialNB()
model.fit(X_train_cnt, y_train)
y_pred = model.predict(X_test_cnt)
print("classification:", classification_report(y_pred, y_test))

tdidf = TfidfVectorizer()

X_train_cnt_td = tdidf.fit_transform(X_train)
X_test_cnt_td = tdidf.transform(X_test)
model_td = MultinomialNB()
model_td.fit(X_train_cnt_td, y_train)
y_pred_td = model_td.predict(X_test_cnt_td)
print("classification_td:", classification_report(y_pred_td, y_test))

classification:               precision    recall  f1-score   support

           0       0.91      0.96      0.94       111
           1       0.95      0.89      0.92        89

    accuracy                           0.93       200
   macro avg       0.93      0.93      0.93       200
weighted avg       0.93      0.93      0.93       200

classification_td:               precision    recall  f1-score   support

           0       0.88      0.96      0.92       107
           1       0.95      0.85      0.90        93

    accuracy                           0.91       200
   macro avg       0.92      0.91      0.91       200
weighted avg       0.91      0.91      0.91       200

